# Indexing

Heißt das so? Die Daten müssen jetzt in die Datenbank.

- DB: ChromaDB

In [ ]:
import json
import chromadb

from chromadb.config import Settings

In [4]:
# ChromaDB initiallisieren
client = chromadb.PersistentClient(path="../data/vector_store")

# Collection erstellen

# client.delete_collection(name="ProduktRAG")
collection = client.get_or_create_collection(
    name="ProduktRAG",
    metadata={"description": "Collection mit allen Beschreibungen und techn. Daten für das ProduktRAG"}
)

## Dataprep

Daten laden und für die DB aufbereiten. Das Schema sieht wie folgt aus:

```python
collection.add(
    documents=[...]
    metadatas=[...]
    ids=[...]
)
```

In [5]:
# Chunks laden
chunks = []

with open("../data/processed/products_embedded.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

print(f"{len(chunks)} Chunks geladen")

5293 Chunks geladen


In [12]:
# Chroma-Schema bauen
documents, embeddings, metadatas, ids = [], [], [], []

for i, chunk in enumerate(chunks):
    documents.append(chunk['document'])
    embeddings.append(chunk['embedding'])
    ids.append(chunk['id'])
    metadatas.append(chunk['metadata'])

## Daten in die DB bringen

In [ ]:
# Daten speichern
collection.add(
    ids=ids,
    embeddings=embeddings,
    metadatas=metadatas,
    documents=documents
)

print(f"{collection.count()} Chunks in der DB")

=== Debug vor collection.add() ===
len(ids): 5293
len(embeddings): 5293
len(metadatas): 5293
len(documents): 5293

Type checks:
type(ids): <class 'list'>
type(embeddings): <class 'list'>
type(metadatas): <class 'list'>
type(documents): <class 'list'>

First elements:
ids[0]: Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank_desc_00
embeddings[0] length: 1024
metadatas[0]: {'product_id': 'Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank', 'title': 'Kirsch LABO-288 PRO-ACTIVE Laborkühlschrank', 'chunk_type': 'desc'}
documents[0][:50]: Der Kirsch LABO-288 ist ein Laborkühlschrank, der 
5293 Chunks in der DB


## Evaluation

Schauen, ob alles geklappt hat. Also Collection auslesen z.B. oder einen bestimmten Chunk auslesen und gegen die chunked-File prüfen.